In [1]:
import pandas as pd

In [2]:
df_full = pd.read_csv('data/full/hotels_with_clusters.csv', encoding="utf-8")
df_full.head(1)

,location_id,address_obj.state,name_details,latitude,longitude,rating,num_reviews,photo_count,price_level,amenities,...,ranking_ratio,price_level_num,bad_review_share,excellent_review_share,location_rating,rooms_rating,service_rating,value_rating,cleanliness_rating,subratings_confidence
0,278399,Podlaskie Province,Hotel Cristal Bialystok,53.132675,23.15599,4.5,390,221,$,"['Internet', 'Suites', 'Room service', 'Free I...",...,0.0625,1.0,0.015385,0.587179,4.8,4.3,4.7,4.3,4.6,0.5


In [3]:
df = df_full[
    [
        "location_id",
        "name_details",
        "address_obj.city_details",
        "rating",
        "price_level",
        "amenities",
        "styles",
        "description",
    ]
].copy()

df["amenities"] = df["amenities"].fillna("")
df["styles"] = df["styles"].fillna("")
df["description"] = df["description"].fillna("")

df["text"] = (
    df["name_details"].fillna("").astype(str) + " " +
    df["address_obj.city_details"].fillna("").astype(str) + " " +
    df["description"].fillna("").astype(str) + " " +
    df["amenities"].fillna("").astype(str) + " " +
    df["styles"].fillna("").astype(str)
)

df.head(2)

,location_id,name_details,address_obj.city_details,rating,price_level,amenities,styles,description,text
0,278399,Hotel Cristal Bialystok,Bialystok,4.5,$,"['Internet', 'Suites', 'Room service', 'Free I...","['Modern', 'Classic']",,Hotel Cristal Bialystok Bialystok ['Internet'...
1,10050636,Ibis Styles Bialystok,Bialystok,4.5,$,"['Internet', 'Kids Activities', 'Suites', 'Roo...","['Modern', 'Mid-range']",,"Ibis Styles Bialystok Bialystok ['Internet', ..."


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = df['text'].tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=30000,
    min_df=1,
    sublinear_tf=True,
    stop_words='english'
)

X = vectorizer.fit_transform(texts)

print(X.shape)

(135, 9234)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def hotel_recomendation(location_id, k = 5):
    idx_list = df.index[df['location_id'] == location_id].tolist()

    idx = idx_list[0]
    scores = cosine_similarity(X[idx], X).flatten()
    best = scores.argsort()[::-1]

    recommendations = []

    for i in best:
        if i == idx:
            continue
        recommendations.append({
            "location_id": int(df.iloc[i]["location_id"]),
            "name": df.iloc[i]["name_details"],
            "score": float(scores[i]),
            "description": df.iloc[i]["description"][:300],
            "amenities": df.iloc[i]["amenities"],
            "styles": df.iloc[i]["styles"],
        })

        if len(recommendations) == k:
            break

    return recommendations

In [6]:
def search_hotels(query, k = 5):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).flatten()

    best = scores.argsort()[::-1][:k]
    
    results = []

    for i in best:
        results.append({
            "location_id": int(df.iloc[i]["location_id"]),
            "name": df.iloc[i]["name_details"],
            "score": float(scores[i]),
            "description": df.iloc[i]["description"][:300],
            "amenities": df.iloc[i]["amenities"],
            "styles": df.iloc[i]["styles"],
        })
    return results

In [7]:
import numpy as np

df = df.replace({np.nan: None})

hotels_metadata = df[
    [
        "location_id",
        "name_details",
        "address_obj.city_details",
        "rating",
        "price_level",
        "description",
        "amenities",
        "styles",
    ]
].to_dict(orient="records")

import joblib
from pathlib import Path

MODEL_DIR = Path("models/tf_idf")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

index_data = {
    "vectorizer": vectorizer,
    "matrix": X,
    "hotels": hotels_metadata,
}

joblib.dump(index_data, MODEL_DIR / "tfidf_index.joblib")
print(f"Zapisano indeks TF-IDF: {MODEL_DIR}/tfidf_index.joblib")


Zapisano indeks TF-IDF: models\tf_idf/tfidf_index.joblib


In [8]:
hotel_id = df.iloc[0]["location_id"]
print(hotel_id)
hotel_recomendation(hotel_id, k=3)

278399


[{'location_id': 21356073,
  'name': 'Mercure Bialystok',
  'score': 0.38820538243524,
  'description': '',
  'amenities': '[\'Shuttle Bus Service\', \'Suites\', \'Room service\', \'Free Internet\', \'Wheelchair access\', \'Restaurant\', \'Bar/Lounge\', \'Pets Allowed\', \'Spa\', \'Public Wifi\', \'Free Wifi\', \'Meeting rooms\', \'Non-smoking rooms\', \'Business center\', \'Fitness center\', \'Laundry Service\', \'Concierge\', \'Banquet Room\', \'Air conditioning\', \'Accessible rooms\', \'Minibar\', \'Conference Facilities\', \'Non-smoking hotel\', \'Safe\', \'Flatscreen TV\', \'Breakfast Buffet\', \'Breakfast Available\', \'Housekeeping\', \'Baggage Storage\', \'Bathrobes\', \'Blackout Curtains\', \'Bottled Water\', \'Breakfast in the Room\', \'Cable / Satellite TV\', \'24-Hour Check-in\', \'Express Check-in / Check-out\', "Children\'s Television Networks", \'Clothes Rack\', \'Desk\', \'Electric Kettle\', \'English\', \'First Aid Kit\', \'Complimentary Toiletries\', \'24-Hour Front 

In [9]:
hotel_id = df.iloc[1]["location_id"]
hotel_recomendation(hotel_id, k=3)


[{'location_id': 14958792,
  'name': 'Ibis Poznan Centrum',
  'score': 0.44544472888270914,
  'description': '',
  'amenities': "['Internet', 'Kids Activities', 'Suites', 'Room service', 'Free Internet', 'Wheelchair access', 'Restaurant', 'Bar/Lounge', 'Pets Allowed', 'Casino', 'Wifi', 'Public Wifi', 'Free Wifi', 'Breakfast included', 'Meeting rooms', 'Non-smoking rooms', 'Business center', 'Fitness center', 'Laundry Service', 'Banquet Room', 'Air conditioning', 'Family Rooms', 'Multilingual Staff', 'Accessible rooms', 'Conference Facilities', 'Non-smoking hotel', 'Breakfast Buffet', 'Breakfast Available', 'Baggage Storage', '24-Hour Front Desk', 'Polish', 'Paid Private Parking On-site']",
  'styles': "['Family', 'Mid-range']"},
 {'location_id': 1885452,
  'name': 'B&B Hotel Torun',
  'score': 0.3572330839268294,
  'description': '',
  'amenities': "['Internet', 'Free Internet', 'Wheelchair access', 'Pets Allowed', 'Wifi', 'Public Wifi', 'Free Wifi', 'Non-smoking rooms', 'Air condition

In [10]:
hotel_id = df.iloc[2]["location_id"]
hotel_recomendation(hotel_id, k=3)

[{'location_id': 15243198,
  'name': 'Hampton By Hilton Poznan Old Town',
  'score': 0.3383477661100848,
  'description': "Hampton by Hilton Poznan Old Town offers a modern and comfortable stay in the heart of one of Poland’s most historic cities. Located just steps from the Old Market Square, the National Museum, and the city's Royal Castle. Located 10 minutes from the Old Zoo and performances at the Poznan Grand Theat",
  'amenities': "['Internet', 'Free Internet', 'Wheelchair access', 'Bar/Lounge', 'Free Wifi', 'Breakfast included', 'Dry Cleaning', 'Non-smoking rooms', 'Fitness center', 'Laundry Service', 'Air conditioning', 'Non-smoking hotel', 'Safe', 'Breakfast Buffet', 'Baggage Storage', 'Blackout Curtains', 'English', '24-Hour Front Desk', 'Polish', 'Soundproof Rooms', 'Meeting rooms', 'Business center', 'Parking', 'Walk-in Shower']",
  'styles': "['Family', 'Mid-range']"},
 {'location_id': 17678224,
  'name': 'Wynajem Pokoi Bialystok',
  'score': 0.10492064320345462,
  'descri

In [11]:
search_hotels("activity for children", k=3
              )

[{'location_id': 277647,
  'name': 'Novotel Gdansk Centrum',
  'score': 0.06234651784003262,
  'description': "Novotel Gdansk Centrum is an ideal base for exploring Gdańsk's Old Town. Literally 5 minutes’ walk from Long Market with beautiful sights - The Neptune fountain, the Town Hall, the nearby Crane, the St. Mary's Church and all the unique atmosphere of Gdansk in its most sought after editions, all with",
  'amenities': '[\'Restaurant\', \'Internet\', \'Room service\', \'Free Internet\', \'Wheelchair access\', \'Bar/Lounge\', \'Wifi\', \'Free Wifi\', \'Kids Activities\', \'Fitness center\', \'Dry Cleaning\', \'Meeting rooms\', \'Non-smoking rooms\', \'Business center\', \'Laundry Service\', \'Air conditioning\', \'Family Rooms\', \'Multilingual Staff\', \'Accessible rooms\', \'Conference Facilities\', \'Non-smoking hotel\', \'Safe\', \'Flatscreen TV\', \'Breakfast Buffet\', \'Parking\', \'Facilities for Disabled Guests\', \'Housekeeping\', \'Bath / Shower\', \'Bottled Water\', \'2

In [12]:
search_hotels("free internet", k=3)

[{'location_id': 5323609,
  'name': 'Noclegi Bydgoszcz - Centrum Onkologii',
  'score': 0.1812754210392287,
  'description': '',
  'amenities': "['Internet', 'Free Internet', 'Wifi', 'Free Wifi']",
  'styles': '[]'},
 {'location_id': 1885452,
  'name': 'B&B Hotel Torun',
  'score': 0.10938220433420558,
  'description': '',
  'amenities': "['Internet', 'Free Internet', 'Wheelchair access', 'Pets Allowed', 'Wifi', 'Public Wifi', 'Free Wifi', 'Non-smoking rooms', 'Air conditioning', 'Multilingual Staff', 'Smoking rooms available', 'Accessible rooms', 'Non-smoking hotel', 'Flatscreen TV', 'Breakfast Buffet', 'Breakfast Available', 'Parking', 'Baggage Storage', 'Bath / Shower', 'Clothes Rack', 'Desk', 'English', '24-Hour Front Desk', 'Hair Dryer', 'Polish', 'Paid Private Parking On-site', 'Soundproof Rooms', 'Telephone', 'Wake Up Service / Alarm Clock']",
  'styles': "['Modern', 'Business']"},
 {'location_id': 14958792,
  'name': 'Ibis Poznan Centrum',
  'score': 0.10931794978598883,
  'des